# Actividad 1: carga y preparación del dataset Titanic

Este notebook coordina las clases encargadas de obtener los datos desde Hugging Face y generar su resumen descriptivo.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / "AE1").exists():
    project_root = project_root.parents[1]
sys.path.insert(0, str(project_root))

from AE1.src.dataset_summarizer import DatasetSummarizer
from AE1.src.hf_dataset_loader import HFDatasetLoader

results_dir = project_root / "AE1" / "resultados"
results_dir.mkdir(parents=True, exist_ok=True)

## 1. Carga del dataset

Se utiliza el dataset público `BIT/titanic-dataset` y su split `train`. La conexión, descarga y conversión a DataFrame se realizan dentro de `HFDatasetLoader`.

In [ ]:
loader = HFDatasetLoader(dataset_id="BIT/titanic-dataset", split="train")
titanic_df = loader.load()
print(f"DataFrame original: {titanic_df.shape[0]} filas y {titanic_df.shape[1]} columnas")
titanic_df.head()

## 2. Transformación mínima

Se seleccionan seis columnas relevantes para el análisis y se renombran al español. La operación produce un nuevo DataFrame que será utilizado para generar el resumen.

In [ ]:
columnas_seleccionadas = ["survived", "pclass", "sex", "age", "fare", "embark_town"]
nombres_en_espanol = {
    "survived": "sobrevivio",
    "pclass": "clase_pasajero",
    "sex": "sexo",
    "age": "edad",
    "fare": "tarifa",
    "embark_town": "ciudad_embarque",
}

titanic_transformado_df = (
    titanic_df[columnas_seleccionadas]
    .rename(columns=nombres_en_espanol)
    .copy()
)

print(
    f"DataFrame transformado: {titanic_transformado_df.shape[0]} filas y "
    f"{titanic_transformado_df.shape[1]} columnas"
)
titanic_transformado_df.head()

## 3. Generación del resumen

El DataFrame transformado se entrega a `DatasetSummarizer`. El archivo resultante contiene dimensiones, nombres, tipos, valores nulos y estadísticas descriptivas.

In [ ]:
summarizer = DatasetSummarizer(titanic_transformado_df)
csv_path = results_dir / "titanic_transformado.csv"
titanic_transformado_df.to_csv(csv_path, index=False, encoding="utf-8")
summary_path = summarizer.save(results_dir / "resumen_titanic.txt")
summary_text = summary_path.read_text(encoding="utf-8")
print(f"Dataset transformado guardado en: {csv_path.relative_to(project_root)}")
print(f"Resumen guardado en: {summary_path.relative_to(project_root)}")
print(summary_text)